In [1]:
import os
import random
import logging
import gc

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight

import torch
torch.set_num_threads(4)

from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
SEED = 42

TRAIN_PATH = "jigsaw-toxic-comment-train.csv"
VAL_PATH = "validation-processed-seqlen128.csv"
MAX_SAMPLES = 1000
EPOCHS = 1
OUTPUT_DIR = "./output/mbert"


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def load_data(path: str):
    df = pd.read_csv(path, on_bad_lines="skip", engine="python", encoding="utf-8")
    df["comment_text"] = df["comment_text"].fillna("")
    return df


def make_binary_label(df: pd.DataFrame):
    toxic_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
    if all(c in df.columns for c in toxic_cols):
        return (df[toxic_cols].sum(axis=1) > 0).astype(int).tolist()
    if "toxic" in df.columns:
        return df["toxic"].astype(int).tolist()
    raise ValueError("No toxic labels found")


class ToxicDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    set_seed(SEED)

    train_df = load_data(TRAIN_PATH)
    val_df = load_data(VAL_PATH)
    if MAX_SAMPLES is not None:
        train_df = train_df.sample(n=MAX_SAMPLES, random_state=SEED)

    train_texts = train_df["comment_text"].astype(str).tolist()
    train_labels = make_binary_label(train_df)
    val_texts = val_df["comment_text"].astype(str).tolist()
    val_labels = make_binary_label(val_df)

    logger.info("Train: %d, Val: %d, Toxic ratio: %.3f", len(train_texts), len(val_texts), sum(train_labels)/len(train_labels))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    logger.info("Tokenizing...")
    train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
    val_enc = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")

    train_dataset = ToxicDataset(train_enc, train_labels)
    val_dataset = ToxicDataset(val_enc, val_labels)

    device = torch.device("cpu")
    logger.info("Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    best_f1 = 0.0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        logger.info("Epoch %d/%d — Training %d batches", epoch, EPOCHS, len(train_loader))
        pbar = tqdm(train_loader, desc=f"Train {epoch}", leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = nn.CrossEntropyLoss(weight=class_weights_tensor)(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        logger.info("Epoch %d loss: %.4f", epoch, total_loss / len(train_loader))

        model.eval()
        all_preds, all_labels, all_probs = [], [], []
        logger.info("Validating...")
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val {epoch}", leave=False):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=1)[:, 1]
                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
                all_probs.extend(probs.cpu().tolist())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, pos_label=1)
        logger.info("Epoch %d — acc: %.4f, f1: %.4f", epoch, acc, f1)

        if f1 > best_f1:
            best_f1 = f1
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
            logger.info("Saved best model")

    if all_probs:
        precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        preds_tuned = (np.array(all_probs) >= best_threshold).astype(int)
        f1_tuned = f1_score(all_labels, preds_tuned, pos_label=1)
        logger.info("Best threshold: %.3f, tuned F1: %.4f", best_threshold, f1_tuned)

    logger.info("Best F1: %.4f", best_f1)


if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]